In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML

cwd = Path.cwd()
project = next(
    candidate
    for parent in (cwd, *cwd.parents)
    for candidate in (parent, parent / "experiments_CausalRL")
    if (candidate / "re2_investigation/data/cases.parquet").is_file()
)
sys.path.insert(0, str(project))
if (project / ".deps").is_dir():
    sys.path.insert(0, str(project / ".deps"))

from common import case_index
from re2_investigation.agent import M, Investigation
from re2_investigation.data import DATA, DATASET, METRIC_LABELS, SERVICES, SOURCES, WIDTH, prepare_case

cases = case_index(DATA, DATASET).sample(frac=1, random_state=471)
cases.index = pd.Index([f"OB-{i:03d}" for i in range(1, len(cases) + 1)], name="Incident")
cases["Split"] = cases["repetition"].map({1: "Training", 2: "Validation", 3: "Final holdout"})

def aligned(frame):
    return HTML(
        "<style>.rca-table thead th { text-align: center; } "
        ".rca-table tbody th, .rca-table tbody td { text-align: left; }</style>"
        + frame.to_html(classes="rca-table", justify="center", float_format=lambda value: f"{value:.3f}")
    )


In [2]:
aligned(
    cases.groupby("Split").agg(
        Incidents=("case", "size"),
        Root_services=("root_cause_service", "nunique"),
        Fault_types=("fault", "nunique"),
    ).loc[["Training", "Validation", "Final holdout"]]
)

,Incidents,Root_services,Fault_types
Split,,,
Training,30,5,6
Validation,30,5,6
Final holdout,30,5,6


In [3]:
holdout = cases.loc["OB-051"]
training = cases.loc[
    cases["repetition"].eq(1)
    & cases["root_cause_service"].eq(holdout["root_cause_service"])
    & cases["fault"].eq(holdout["fault"])
].iloc[0]
examples = {"Training": training, "Final holdout": holdout}

aligned(
    pd.DataFrame(
        {split: {"Incident": row.name, "Repetition": row["repetition"]} for split, row in examples.items()}
    )
)

,Training,Final holdout
Incident,OB-054,OB-051
Repetition,1,3


**Starting input**

Before any query, the agent sees how frontend metrics changed between the two minutes before and the two minutes after fault injection.

In [4]:
envs = {}
initial = {}
training_calls = {}
for split, row in examples.items():
    case, calls, _ = prepare_case(DATA / row["case"], row.name, int(row["repetition"]), row["inject_time"])
    if split == "Training":
        training_calls = calls
    roots = {row.name: row["root_cause_service"]} if split == "Training" else None
    envs[split] = Investigation([case], training_roots=roots)
    initial[split], _ = envs[split].reset(options={"index": 0, "budget": 4})


def metric_inputs(observations, start, label):
    columns = {}
    for split, observation in observations.items():
        change, present = observation[start : start + WIDTH].reshape(-1, 2).T
        columns[(split, "Scaled change")] = change.round(3)
        columns[(split, "Available")] = present.astype(bool)
    return pd.DataFrame(columns, index=pd.Index(METRIC_LABELS, name=label))


aligned(metric_inputs(initial, 0, "Starting input: frontend metric"))

**Service graph example**

Each row shows which service called another service and how many calls were recorded in the two minutes before fault injection in training incident OB-054. The graph-guided agent uses the connections as clues about where to look.

In [5]:
graph_services = {"frontend", *SERVICES}
graph_example = pd.DataFrame(
    [
        (caller, callee, count)
        for (caller, callee), count in sorted(training_calls.items())
        if caller in graph_services and callee in graph_services
    ],
    columns=["Caller", "Callee", f"Calls before fault in {training.name}"],
)
aligned(graph_example)

,Caller,Callee,Calls before fault in OB-054
0,checkoutservice,currencyservice,253
1,checkoutservice,emailservice,75
2,checkoutservice,productcatalogservice,178
3,frontend,checkoutservice,75
4,frontend,currencyservice,4488
5,frontend,productcatalogservice,7514
6,frontend,recommendationservice,1157
7,recommendationservice,productcatalogservice,1157


**New input**

After querying `emailservice` metrics, the agent sees that service’s changes over the same two time windows. This evidence is added to the frontend data it already saw.

In [6]:
query = SERVICES.index("emailservice") * M + SOURCES.index("Metrics")
after_query = {split: env.step(query)[0] for split, env in envs.items()}

aligned(metric_inputs(after_query, WIDTH + query * WIDTH, "New input: emailservice metric"))

**Answer key**

RCAEval records which service was faulted and what type of fault was injected. The agent never sees these labels as input. The training environment uses the known root service to reward guesses, and the final holdout label is used only to grade the trained agent.

In [7]:
aligned(
    pd.DataFrame(
        {
            split: {
                "Known root service": row["root_cause_service"],
                "Injected fault": row["fault"],
                "Label used for": "Training reward" if split == "Training" else "Final grading",
            }
            for split, row in examples.items()
        }
    ).rename_axis("Answer key, separate from model input")
)

,Training,Final holdout
"Answer key, separate from model input",,
Known root service,checkoutservice,checkoutservice
Injected fault,mem,mem
Label used for,Training reward,Final grading
